# Peterson et al. (2021), eNeuro: regression of single-event power on behaviour (Figure 6)

Paper: Peterson, Singh, Wang, Rao and Brunton (2021), eNeuro 8(3), doi:10.1523/ENEURO.0007-21.2021.
Dataset: DANDI:000055, version 0.220127.0436, the dandiset's only published version (see the
header of `reproduce_peterson_2021.ipynb`, which also holds the data check, the link claims and
the clock of the ECoG). Claims: `claims/000055/10.1523-eneuro.0007-21.2021.yaml`.

This notebook tests the six claims on the regression models of Figure 6. It reads the ECoG of
every event again (band power only) and fits, for every electrode and band, the models the
claims file's opening comment describes: forward selection by OLS adjusted R2, sklearn's
HuberRegressor, 200 random splits with 90% training events and equal numbers of test events from
each day, Delta R2 by shuffling one feature. The events are those of reading (b1) in the first
notebook (Figshare false positives dropped, segments rejected on the PSD averaged over good
electrodes). The ECoG clock of the three late-starting files is shifted as in the first notebook.

| Claim | Here |
|---|---|
| fig6b-best-models-sensorimotor | tested |
| fig6-r2-at-most-025 | tested |
| fig6c-angle-and-day-most-informative | tested |
| extfig6-2-angle-most-retained | tested |
| fig6d-upward-reaches-enhance-pattern | tested |
| fig6d-angle-and-day-largest-coefficients | tested |

Not run, for time: the code's 80/20 split and the median band power as variants (each would
double the fits), the code's log(x + 1) transforms, and the events of reading (b3) of the first
notebook (which also drops the segments with zeroed ECoG; a second set of fits does not fit in
the notebook's 30 minutes). The fits here use the events of reading (b1).

In [ ]:
import os, time, io, zlib
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor
import matplotlib.pyplot as plt
import ajile

T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
SUBJECTS = [int(s) for s in os.environ.get("AJILE_SUBJECTS", ",".join(map(str, range(1, 13)))).split(",")]
A_all = ajile.assets()
A = A_all[A_all.subject.isin(SUBJECTS)].reset_index(drop=True)
with ProcessPoolExecutor(8) as ex:
    meta = dict(zip(A.path, ex.map(ajile.read_meta, A.asset_id)))
fig_files = ajile.figshare_files(ajile.EPOCH_ARTICLE)
fig_names = sorted(n for n in fig_files if int(n[5:7]) in SUBJECTS)
with ProcessPoolExecutor(8) as ex:
    figmeta = dict(zip(fig_names, ex.map(ajile.epoch_metadata, fig_names, [fig_files] * len(fig_names))))
roi_txt, roi_files = ajile.roi_matrices()
ev = ajile.events_table(A, meta, figmeta)
print(f"{len(A)} files, {len(ev)} events, {int(ev.false_pos.sum())} Figshare false positives; loaded in {time.time() - T0:.0f} s")

In [ ]:
W_all = {}
for s in SUBJECTS:
    dens, W = ajile.roi_tables(roi_txt[s])
    W_all[s] = W.iloc[:len(meta[A.path[A.subject == s].iloc[0]]["electrodes"])]
good_all = {}
for s in SUBJECTS:
    g = np.stack([meta[p]["electrodes"].good.to_numpy() for p in A.path[A.subject == s]])
    good_all[s] = g.all(0)
    print(f"S{s:02d}: {g.shape[1]} channels, good on every day {int(g.all(0).sum())}, good on some days only "
          f"{int((g.any(0) & ~g.all(0)).sum())}, never good {int((~g.any(0)).sum())}; events {int((ev.subject == s).sum())}, "
          f"days {sorted(ev.day[ev.subject == s].unique())}")
print("ECoG clock shifted (session start, s) for:", ev.groupby("path").offset.first()[lambda x: x > 0].round(1).to_dict())
print("feature ranges over all events:")
print(ev[["day", "hour", "duration", "magnitude", "angle_folded", "onset_speed", "speech_ratio", "bimanual_ratio",
          "bimanual_overlap", "bimanual_class"]].describe().T[["min", "50%", "max"]].round(3).to_string())

### Band power and model fits

Models are fitted at every electrode, good or not; the main results use the electrodes marked
good on every day of the subject, and the code's inclusion of bad electrodes is a variant. One
model per subject, electrode and band; the seed of each model's splits is fixed from its
subject, electrode and band.

In [ ]:
BANDS = {"LFB": 0, "HFB": 2}        # columns of ajile.BAND_COLS: band mean with baseline (Methods)
subject_files = {s: list(A.path[A.subject == s]) for s in SUBJECTS}
file_jobs = [(p, ev.asset_id[ev.path == p].iloc[0], (ev.start_time - ev.offset)[ev.path == p].to_numpy(), None, True) for p in A.path]
DESIGN = {}


def make_models(s, spec_s):
    A_s = A[A.subject == s].reset_index(drop=True)
    ev_s = ev[ev.subject == s].reset_index(drop=True)
    ok = np.concatenate([spec_s[p]["ok"] for p in A_s.path])
    rej1, _, _, _ = ajile.rejection(ev_s.assign(ok=ok), A_s, meta, spec_s, [s])
    keep = (ev_s.false_pos == 0).to_numpy() & ok & ~rej1
    e = ev_s[keep]
    X, groups = ajile.design(e)
    band = np.concatenate([spec_s[p]["band"] for p in A_s.path])[keep]
    DESIGN[s] = dict(X=X, groups=groups, n=int(keep.sum()), rejected=int((ok & rej1).sum()))
    jobs = []
    for c in range(band.shape[1]):
        for b, j in BANDS.items():
            seed = zlib.crc32(f"{s}-{c}-{b}".encode())
            jobs.append(((s, c, b), X, groups, band[:, c, j], e.day.to_numpy(), seed, {"n_splits": 200, "test_frac": 0.1}))
    return jobs


t = time.time()
spec, fits, timing = ajile.regression_pipeline(file_jobs, subject_files, make_models, workers=12)
print(f"band power for {sum(len(j[2]) for j in file_jobs)} events and {len(fits)} models in {time.time() - t:.0f} s; "
      f"subjects read at (s): { {k: round(v) for k, v in timing['subjects_read_at'].items()} }")
print("events per subject used:", {s: d["n"] for s, d in DESIGN.items()})
print("columns of the design:", {s: len(d["groups"]) for s, d in DESIGN.items()})

In [ ]:
FEATS = ajile.FEATURES
rows = []
for (s, c, b), r in fits.items():
    g = DESIGN[s]["groups"]
    coef = r["coef"]
    cf = {}
    for f in FEATS:
        m = g == f
        cf[f] = np.std(coef[m]) if f in ("day", "time_of_day") else coef[m][0]
    rows.append(dict(subject=s, ch=c, band=b, good=bool(good_all[s][c]), r2=r["r2"], n=r["n"],
                     n_selected=len(r["selected"]), **{"sel_" + f: f in r["selected"] for f in FEATS},
                     **{"dm_" + f: r["delta_methods"][f] for f in FEATS}, **{"dc_" + f: r["delta_code"][f] for f in FEATS},
                     **{"coef_" + f: cf[f] for f in FEATS}))
R = pd.DataFrame(rows)
print(f"models: {len(R)}; at good electrodes {int(R.good.sum())}; with nothing selected {int((R.n_selected == 0).sum())}; "
      f"failed fits {len(DESIGN) and sum(1 for s in SUBJECTS for c in range(len(good_all[s])) for b in BANDS if (s, c, b) not in fits)}")
print("full-model test R2 at good electrodes, by band:")
print(R[R.good].groupby("band").r2.describe().round(3).to_string())
print("R2 range over all models:", R.r2.min().round(3), "to", R.r2.max().round(3))
LFB_R2 = R[R.band == "LFB"].set_index(["subject", "ch"]).r2
R["r2_lfb"] = [LFB_R2[(s, c)] for s, c in zip(R.subject, R.ch)]
print("models with R2 > 0.05, good electrodes:", R[R.good & (R.r2 > 0.05)].groupby("band").size().to_dict())
print("by subject (good, R2 > 0.05):")
print(R[R.good & (R.r2 > 0.05)].groupby(["subject", "band"]).size().unstack(fill_value=0).T.to_string())

> fig6-r2-at-most-025: All regression models explain at most 25% of the variance of withheld
> single-event power.

In [ ]:
best = R[R.good].sort_values("r2", ascending=False).head(5)
print("largest split-averaged R2, good electrodes:")
print(best[["subject", "ch", "band", "r2", "n_selected"]].round(4).to_string(index=False))
print(f"largest R2 at good electrodes: {R[R.good].r2.max():.4f}; at most 0.275: {R[R.good].r2.max() <= 0.275}")
print(f"variant, all electrodes (bad included): {R.r2.max():.4f}")
print("largest per subject and band (good):")
print(R[R.good].groupby(["subject", "band"]).r2.max().unstack().round(3).T.to_string())
el0 = pd.concat([meta[p]["electrodes"].assign(subject=s) for s in SUBJECTS for p in A.path[A.subject == s][:1]])
print("for reference only, the R2 columns stored in the dataset's electrode table (not used for the outcome):",
      "low_freq_R2 max", round(el0.low_freq_R2.max(), 3), "; high_freq_R2 max", round(el0.high_freq_R2.max(), 3))
cmp = R[R.good].pivot_table(index=["subject", "ch"], columns="band", values="r2")
el0 = el0.reset_index().rename(columns={"index": "ch"}).set_index(["subject", "ch"])
j = cmp.join(el0[["low_freq_R2", "high_freq_R2"]])
print("correlation of these R2 with the stored columns (good electrodes): LFB",
      round(j.LFB.corr(j.low_freq_R2), 3), "; HFB", round(j.HFB.corr(j.high_freq_R2), 3))

**Result: reproduced.** The largest split-averaged withheld R2 over good electrodes and both
bands is 0.2626 (S04, HFB), at most 0.275. With electrodes marked bad included it is 1.0000,
from a bad electrode. These R2 correlate with the R2 columns stored in the dataset's electrode
table (0.74 for LFB, 0.706 for HFB).

> fig6b-best-models-sensorimotor: Regression models with the highest withheld R2 are mainly at
> electrodes over frontoparietal sensorimotor areas, for both bands.

(a) Each electrode in the region of the eight with its largest weight in the paper's projection
matrix. (b) R2 projected onto the eight regions with the projection weights (electrodes good on
every day, each region renormalised), then averaged over subjects.

In [ ]:
FP5 = ["Frontal_Mid", "Precentral", "Postcentral", "Parietal_Inf", "SupraMarginal"]
SM3 = ["Precentral", "Postcentral", "Parietal_Inf"]
TEMP = ["Temporal_Sup", "Temporal_Mid", "Temporal_Inf"]
assign, wmax = {}, {}
for s in SUBJECTS:
    W8 = W_all[s][ajile.REGIONS].to_numpy(float)
    for c in range(len(W8)):
        assign[(s, c)] = ajile.REGIONS[int(np.argmax(W8[c]))]
        wmax[(s, c)] = W8[c].max()
R["region"] = [assign[(s, c)] for s, c in zip(R.subject, R.ch)]
R["wmax"] = [wmax[(s, c)] for s, c in zip(R.subject, R.ch)]
for band in BANDS:
    g = R[R.good & (R.band == band)]
    hi = g[g.r2 > 0.05]
    frac = hi.region.isin(FP5).mean() if len(hi) else np.nan
    med_sm, med_t = g[g.region.isin(SM3)].r2.median(), g[g.region.isin(TEMP)].r2.median()
    print(f"{band} (a): electrodes with R2 > 0.05: {len(hi)}, in the five frontoparietal regions {frac:.3f}; "
          f"median R2 sensorimotor {med_sm:.4f} against temporal {med_t:.4f}")
    print("    counts of R2 > 0.05 by region:", hi.region.value_counts().to_dict())
    print("    median R2 by region:", g.groupby("region").r2.median().round(4).to_dict())
    lowk = g[g.wmax < 1e-6]
    print(f"    electrodes whose largest weight is below 1e-6 (assigned anyway): {len(lowk)}; without them, fraction in "
          f"frontoparietal {hi[hi.wmax >= 1e-6].region.isin(FP5).mean():.3f}")
    proj_r2 = []
    for s in SUBJECTS:
        P = ajile.projection(W_all[s], good_all[s])
        r2 = g[g.subject == s].set_index("ch").r2.reindex(range(len(P))).fillna(0).to_numpy()
        proj_r2.append(np.nan_to_num(P, nan=0).T @ r2 / np.where(np.isnan(P).all(0), np.nan, 1))
    pr = pd.Series(np.nanmean(proj_r2, 0), index=ajile.REGIONS).sort_values(ascending=False)
    print(f"{band} (b): projected R2 averaged over subjects:", pr.round(4).to_dict())
    print(f"    top three {list(pr.index[:3])}, all among the five frontoparietal: {set(pr.index[:3]) <= set(FP5)}")

**Result: reproduced.** Under reading (a), all 12 LFB and 6 HFB models with R2 > 0.05 are at
electrodes assigned to the five frontoparietal regions (fraction 1.000), and the median R2 of
sensorimotor electrodes is above that of temporal ones (-0.0172 against -0.0210 for LFB, -0.0185
against -0.0212 for HFB). Under reading (b) the three regions with the highest projected R2 are
precentral, postcentral and middle frontal in both bands. Few models pass 0.05 (18 of 1952 at
good electrodes), from four subjects.

> fig6c-angle-and-day-most-informative: Reach angle and day of recording are the most
> informative features for single-event power, especially for low-frequency power.

In [ ]:
def ranking(df, prefix):
    return df[[prefix + f for f in FEATS]].median().rename(lambda c: c[len(prefix):]).sort_values(ascending=False)


variants = {
    "main (Methods Delta R2, good, R2 > 0.05 per band)": (lambda d, b: d[d.good & (d.band == b) & (d.r2 > 0.05)], "dm_"),
    "code Delta R2 (test events shuffled too)": (lambda d, b: d[d.good & (d.band == b) & (d.r2 > 0.05)], "dc_"),
    "selection by LFB R2 for both bands": (lambda d, b: d[d.good & (d.band == b) & (d.r2_lfb > 0.05)], "dm_"),
    "bad electrodes included": (lambda d, b: d[(d.band == b) & (d.r2 > 0.05)], "dm_"),
}
for name, (fsel, pre) in variants.items():
    for band in BANDS:
        d = fsel(R, band)
        rk = ranking(d, pre)
        print(f"{name}, {band} (n={len(d)}): " + ", ".join(f"{k} {v:.4f}" for k, v in rk.items()))
        print(f"    top two: {list(rk.index[:2])}")
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, band in zip(axs, BANDS):
    d = R[R.good & (R.band == band) & (R.r2 > 0.05)]
    ax.boxplot([d["dm_" + f] for f in FEATS], showfliers=False)
    ax.set_xticks(range(1, 11), FEATS, rotation=60, ha="right")
    ax.set_title(f"{band}, n={len(d)}")
axs[0].set_ylabel("Delta R2")
plt.tight_layout()
plt.show()
vals = R[R.good & (R.r2 > 0.05)][["dm_" + f for f in FEATS]]
print(f"Delta R2 range over the plotted models: {vals.min().min():.4f} to {vals.max().max():.4f} (outliers not drawn)")

**Result: reproduced.** Over the 12 LFB models with R2 > 0.05, reach angle (median Delta R2
0.0823) and day of recording (0.0104) have the two largest medians; over the 6 HFB models day
(0.1708) and reach angle (0.0127) are first and second. The code's Delta R2 gives the same top
two. With selection by LFB R2, or with bad electrodes included, the HFB medians are all 0.

> extfig6-2-angle-most-retained: Reach angle is the feature most often retained by forward
> selection in sensorimotor regions, followed by reach duration, reach magnitude, bimanual ratio
> and bimanual overlap.

In [ ]:
ret = {}
for band in BANDS:
    per_s = []
    for s in SUBJECTS:
        d = R[(R.subject == s) & (R.band == band)].set_index("ch").reindex(range(len(good_all[s])))
        S = d[["sel_" + f for f in FEATS]].astype(float).to_numpy()
        P = np.nan_to_num(ajile.projection(W_all[s], good_all[s]), nan=0)
        valid = P.sum(0) > 0
        v = (P.T @ np.nan_to_num(S)) / np.where(valid, 1, np.nan)[:, None]
        per_s.append(v)
    ret[band] = pd.DataFrame(np.nanmean(per_s, 0), index=ajile.REGIONS, columns=FEATS)
    print(f"{band}: retention probability, projected and averaged over subjects")
    print(ret[band].round(3).to_string())
angle_top = {(b, r): ret[b].loc[r].idxmax() == "reach_angle" for b in BANDS for r in SM3}
print("reach angle highest in each sensorimotor region and band:", angle_top, "->", all(angle_top.values()))
order = ["reach_duration", "reach_magnitude", "bimanual_ratio", "bimanual_overlap"]
for name, regs in (("(a) three sensorimotor regions, both bands", SM3), ("(b) all eight regions, both bands", ajile.REGIONS)):
    avg = pd.concat([ret[b].loc[regs] for b in BANDS]).mean().sort_values(ascending=False)
    print(f"{name}: " + ", ".join(f"{k} {v:.3f}" for k, v in avg.items()))
    print(f"    ranked second to fifth: {list(avg.index[1:5])}; the legend's order: {list(avg.index[1:5]) == order}")

**Result: not reproduced.** Reach angle has the highest retention probability in none of the six
sensorimotor region and band pairs; bimanual ratio is the most retained feature in every one of
them (0.454 to 0.598). Averaged over the three sensorimotor regions and both bands, the order
after bimanual ratio (0.526) is bimanual overlap, reach duration, day and reach angle; over all
eight regions it is reach duration, bimanual overlap, bimanual class and day.

> fig6d-upward-reaches-enhance-pattern: Upward reaches are associated with lower low-frequency
> power and higher high-frequency power.

In [ ]:
for name, fsel in (("main", lambda d, b: d[d.good & (d.band == b) & (d.r2 > 0.05)]),
                   ("selection by LFB R2", lambda d, b: d[d.good & (d.band == b) & (d.r2_lfb > 0.05)]),
                   ("bad electrodes included", lambda d, b: d[(d.band == b) & (d.r2 > 0.05)])):
    out = []
    for band in BANDS:
        d = fsel(R, band)
        out.append(f"{band} median angle coefficient {d.coef_reach_angle.median():+.4f} (n={len(d)}, "
                   f"{(d.coef_reach_angle < 0).mean():.2f} negative, {(d.coef_reach_angle == 0).mean():.2f} zero)")
    print(f"{name}: " + "; ".join(out))
d = R[R.good & (R.r2 > 0.05)]
print("by subject, median angle coefficient (main):")
print(d.groupby(["subject", "band"]).coef_reach_angle.median().unstack().round(4).T.to_string())

**Result: reproduced.** The median reach angle coefficient is -1.3385 over the 12 LFB models
with R2 > 0.05 (all negative) and +0.4067 over the 6 HFB models (none negative). With selection
by LFB R2 the HFB median is 0.0000, and with bad electrodes included both medians are 0.0000.

> fig6d-angle-and-day-largest-coefficients: Reach angle and day of recording have the largest
> coefficient magnitudes among the features, and the SD of the recording-day coefficients is
> large.

In [ ]:
for name, fsel in (("main", lambda d, b: d[d.good & (d.band == b) & (d.r2 > 0.05)]),
                   ("selection by LFB R2", lambda d, b: d[d.good & (d.band == b) & (d.r2_lfb > 0.05)]),
                   ("bad electrodes included", lambda d, b: d[(d.band == b) & (d.r2 > 0.05)])):
    for band in BANDS:
        d = fsel(R, band)
        C = d[["coef_" + f for f in FEATS]].rename(columns=lambda c: c[5:])
        ra = C.abs().median().sort_values(ascending=False)
        rb = C.median().abs().sort_values(ascending=False)
        print(f"{name}, {band} (n={len(d)}):")
        print("   (a) median |coef|: " + ", ".join(f"{k} {v:.4f}" for k, v in ra.items()))
        print("   (b) |median coef|: " + ", ".join(f"{k} {v:.4f}" for k, v in rb.items()))
        print(f"   top two (a) {list(ra.index[:2])}; (b) {list(rb.index[:2])}")

**Result: reproduced.** In both bands and under both readings reach angle and day of recording
have the two largest coefficient magnitudes: LFB 1.3385 and 0.6102, HFB 1.1390 (day) and 0.4067
(angle). With selection by LFB R2 the HFB top two are reach duration and day; with bad
electrodes included, reading (b) gives day and time of day.

## Summary

| Claim | Outcome |
|---|---|
| fig6-r2-at-most-025 | reproduced |
| fig6b-best-models-sensorimotor | reproduced |
| fig6c-angle-and-day-most-informative | reproduced |
| extfig6-2-angle-most-retained | not reproduced |
| fig6d-upward-reaches-enhance-pattern | reproduced |
| fig6d-angle-and-day-largest-coefficients | reproduced |

The other claims are in `reproduce_peterson_2021.ipynb`, with the full table.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the events are those of rejection reading (b1)
only, and the readings (b3) and the code's 80/20 split, median band power and log(x + 1)
transforms were not run, for time; electrodes are those marked good on every day of the
subject (electrodes good on some days only are in the bad-electrode variant); the region
assignment uses the paper's projection matrices. Only 18 of 1952 models at good electrodes
reach a withheld R2 above 0.05, so the claims on those models rest on 12 LFB and 6 HFB models
from four subjects.

What was found about the dataset: the electrode table holds R2 columns (`low_freq_R2`,
`high_freq_R2`) without a description of how they were computed; the R2 here correlate with
them (0.74 and 0.706).

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")